# 08 — Balance Dataset & Run AutoML

Augments Fitzpatrick types 4, 5, 6 to balance the combined dataset
(Fitzpatrick17k + SCIN + PAD-UFES), uploads to GCS, and trains AutoML.

**Combined dataset baseline:**
| Type | Count |
|------|-------|
| 1 | 3,302 |
| 2 | 6,474 |
| 3 | 4,029 |
| 4 | 2,464 |
| 5 | 1,258 |
| 6 | 492 |

**Augmentation (Albumentations):** flips, rotations, random crop, Gaussian noise
**Forbidden:** brightness, contrast, color jitter, blurring

In [ ]:
# Cell 1: Setup
import os, subprocess, sys

if not os.path.exists("/content/NST_Class"):
    subprocess.run(["git", "clone", "https://github.com/AayushBaniya2006/NST_Class.git"], cwd="/content")
else:
    subprocess.run(["git", "pull"], cwd="/content/NST_Class")
os.chdir("/content/NST_Class")

!pip install -q -r requirements.txt
!pip install -q albumentations
sys.path.insert(0, "/content/NST_Class")

print("Setup complete!")

In [ ]:
# Cell 2: Authenticate with Google Cloud
from google.colab import auth
auth.authenticate_user()
print("Authenticated!")

In [ ]:
# Cell 3: Configuration
import pandas as pd
import numpy as np

BUCKET_NAME = "skin-tone-project"
GCS_IMAGE_PREFIX = f"gs://{BUCKET_NAME}/combined_images"
LOCAL_IMAGE_DIR = "data/combined_images"
LOCAL_AUG_DIR = "data/augmented_images"

SEED = 42
IMAGE_SIZE = 224

# Augmentation targets per class (professor guidance: 2500-4000 for types 4/5/6)
# We'll augment to bring each to ~3000
# Note: Type 2 remains at 6474 (dominant class) — not downsampled per professor guidance
AUGMENTATION_TARGETS = {
    4: 1000,   # 2464 + 1000 = 3464
    5: 2000,   # 1258 + 2000 = 3258
    6: 2500,   # 492 + 2500 = 2992
}

df = pd.read_csv("combined_dataset.csv")
print("Combined dataset:")
print(df["fitzpatrick_scale"].value_counts().sort_index())
print(f"\nTotal: {len(df)}")
print(f"\nAugmentation plan:")
for cls, count in sorted(AUGMENTATION_TARGETS.items()):
    current = len(df[df["fitzpatrick_scale"] == cls])
    print(f"  Type {cls}: {current} → {current + count} (+{count} augmented)")

In [ ]:
# Cell 4: Download all images from GCS
# This assumes all 18k images have been uploaded to gs://skin-tone-project/combined_images/
# If not yet uploaded, you need to upload them first before running this cell.

os.makedirs(LOCAL_IMAGE_DIR, exist_ok=True)

# Use gcloud storage for fast parallel download (-n = skip existing)
!gcloud storage cp -n "{GCS_IMAGE_PREFIX}/**" {LOCAL_IMAGE_DIR}/

# Count what we have locally
from pathlib import Path
local_images = [f for f in Path(LOCAL_IMAGE_DIR).rglob("*") if f.is_file()]
print(f"Local images: {len(local_images)}")

# Check how many from our CSV we can find
found = 0
missing = []
for img_id in df["img_id"]:
    if (Path(LOCAL_IMAGE_DIR) / img_id).exists():
        found += 1
    else:
        missing.append(img_id)

print(f"Found: {found}/{len(df)}")
if missing:
    print(f"Missing: {len(missing)} images")
    print("First 10 missing:", missing[:10])

In [ ]:
# Cell 5: Augment minority classes
from scripts.augment_minority import augment_images, verify_augmentation

all_augmented = []

for fitz_class, target_count in sorted(AUGMENTATION_TARGETS.items()):
    print(f"\n{'='*50}")
    print(f"Augmenting Fitzpatrick type {fitz_class}")
    print(f"{'='*50}")

    class_imgs = df[df["fitzpatrick_scale"] == fitz_class]["img_id"].tolist()
    print(f"Source images: {len(class_imgs)}")
    print(f"Target new images: {target_count}")

    # Type 6 note: 2500 augmented from 492 originals = ~5x ratio.
    # Monitor for overfitting on type 6 specifically.
    class_aug_dir = f"{LOCAL_AUG_DIR}/type_{fitz_class}"
    created, paths = augment_images(
        input_dir=LOCAL_IMAGE_DIR,
        output_dir=class_aug_dir,
        img_ids=class_imgs,
        target_count=target_count,
        seed=SEED + fitz_class,  # different seed per class
        image_size=IMAGE_SIZE,
    )

    print(f"Created: {created} augmented images")

    # Verify
    result = verify_augmentation(LOCAL_IMAGE_DIR, class_aug_dir)
    print(f"Verification: {result['valid']}/{result['checked']} valid (passed={result['passed']})")
    assert result["passed"], f"Verification failed for type {fitz_class}: {result}"

    all_augmented.extend([
        {"img_id": p.name, "fitzpatrick_scale": fitz_class, "source": "augmented"}
        for p in paths
    ])

print(f"\nTotal augmented: {len(all_augmented)}")

In [ ]:
# Cell 6: Visual spot-check — randomly inspect augmented images for artifacts
import matplotlib.pyplot as plt
from PIL import Image as PILImage

fig, axes = plt.subplots(len(AUGMENTATION_TARGETS), 3, figsize=(12, 4 * len(AUGMENTATION_TARGETS)))

for row, fitz_class in enumerate(sorted(AUGMENTATION_TARGETS.keys())):
    class_aug_dir = Path(f"{LOCAL_AUG_DIR}/type_{fitz_class}")
    aug_files = sorted(class_aug_dir.glob("*"))[:3]

    for col, img_path in enumerate(aug_files):
        ax = axes[row][col]
        ax.imshow(PILImage.open(img_path))
        ax.set_title(f"Type {fitz_class}: {img_path.name}", fontsize=8)
        ax.axis("off")

plt.suptitle("Augmented Image Spot-Check (verify no artifacts)", fontsize=14)
plt.tight_layout()
plt.show()

print("Visually inspect the images above.")
print("If they look wrong, adjust augmentation parameters and re-run cell 5.")

In [ ]:
# Cell 7: Build balanced CSV
aug_df = pd.DataFrame(all_augmented)
balanced_df = pd.concat([df, aug_df], ignore_index=True)

print("Balanced dataset:")
print(balanced_df["fitzpatrick_scale"].value_counts().sort_index())
print(f"\nTotal: {len(balanced_df)}")
print(f"  Original: {len(df)}")
print(f"  Augmented: {len(aug_df)}")

# Save
balanced_df.to_csv("balanced_dataset.csv", index=False)
print("\nSaved to balanced_dataset.csv")

In [ ]:
# Cell 8: Upload augmented images + balanced CSV to GCS
# Using gcloud storage (Go-based, faster than gsutil for bulk transfers)
for fitz_class in sorted(AUGMENTATION_TARGETS.keys()):
    class_aug_dir = f"{LOCAL_AUG_DIR}/type_{fitz_class}"
    print(f"Uploading type {fitz_class} augmented images...")
    !gcloud storage cp -r {class_aug_dir}/* "{GCS_IMAGE_PREFIX}/"

print("Augmented images uploaded!")

# Upload the balanced CSV
!gcloud storage cp balanced_dataset.csv "gs://{BUCKET_NAME}/balanced_dataset.csv"
print("Balanced CSV uploaded!")

In [ ]:
# Cell 9: Generate AutoML manifest with stratified split
# Note: We build the manifest inline rather than using src/utils/gcs.generate_automl_csv()
# because the combined CSV uses different column names (img_id, fitzpatrick_scale)
# than what generate_automl_csv expects (hasher, skin_tone_label).
from sklearn.model_selection import train_test_split

# Stratified split on balanced dataset
train_df, temp_df = train_test_split(
    balanced_df, test_size=0.30, random_state=SEED,
    stratify=balanced_df["fitzpatrick_scale"],
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, random_state=SEED,
    stratify=temp_df["fitzpatrick_scale"],
)

train_df = train_df.copy()
val_df = val_df.copy()
test_df = test_df.copy()
train_df["split"] = "TRAINING"
val_df["split"] = "VALIDATION"
test_df["split"] = "TEST"

manifest_df = pd.concat([train_df, val_df, test_df], ignore_index=True)

# Build AutoML manifest: ML_USE, GCS_FILE_PATH, LABEL
rows = []
for _, row in manifest_df.iterrows():
    gcs_path = f"{GCS_IMAGE_PREFIX}/{row['img_id']}"
    rows.append(f"{row['split']},{gcs_path},{int(row['fitzpatrick_scale'])}")

manifest_path = "data/automl_balanced_manifest.csv"
with open(manifest_path, "w", encoding="utf-8") as f:
    f.write("\n".join(rows) + "\n")

print(f"Manifest: {manifest_path} ({len(rows)} rows)")
print(f"\nSplit distribution:")
print(f"  Train: {len(train_df)}")
print(f"  Val:   {len(val_df)}")
print(f"  Test:  {len(test_df)}")
print(f"\nPer-class in train:")
print(train_df["fitzpatrick_scale"].value_counts().sort_index())

# Upload manifest
!gcloud storage cp {manifest_path} "gs://{BUCKET_NAME}/automl/balanced_manifest.csv"
print("\nManifest uploaded!")

In [ ]:
# Cell 10: Run AutoML on balanced dataset
from google.cloud import aiplatform

PROJECT_ID = "YOUR_PROJECT_ID"  # <-- FILL THIS IN
REGION = "us-central1"

aiplatform.init(project=PROJECT_ID, location=REGION)

# Create dataset from manifest
dataset = aiplatform.ImageDataset.create(
    display_name="skin-tone-balanced",
    gcs_source=f"gs://{BUCKET_NAME}/automl/balanced_manifest.csv",
    import_schema_uri=aiplatform.schema.dataset.ioformat.image.single_label_classification,
)
print(f"Dataset created: {dataset.resource_name}")

# Train
job = aiplatform.AutoMLImageTrainingJob(
    display_name="skin-tone-balanced-automl",
    prediction_type="classification",
    multi_label=False,
    model_type="CLOUD",
)

model = job.run(
    dataset=dataset,
    model_display_name="skin-tone-balanced-v1",
    budget_milli_node_hours=8000,
)
print(f"Model trained: {model.resource_name}")

In [ ]:
# Cell 11: Evaluate AutoML model
model_eval = model.list_model_evaluations()[0]
metrics = model_eval.metrics

print("AutoML Evaluation (Balanced Dataset):")
for key, value in metrics.items():
    print(f"  {key}: {value}")

if "confusionMatrix" in metrics:
    print("\nConfusion Matrix:")
    print(metrics["confusionMatrix"])

print(f"\nModel ID: {model.resource_name}")

In [ ]:
# Cell 12: Summary
print("=" * 60)
print("BALANCED AUGMENTATION SUMMARY")
print("=" * 60)
print(f"\nOriginal dataset: {len(df)} images")
print(f"Augmented images: {len(aug_df)}")
print(f"Balanced total:   {len(balanced_df)}")
print(f"\nPer-class distribution:")
for cls in range(1, 7):
    orig = len(df[df["fitzpatrick_scale"] == cls])
    total = len(balanced_df[balanced_df["fitzpatrick_scale"] == cls])
    aug = total - orig
    print(f"  Type {cls}: {orig} original + {aug} augmented = {total}")
print(f"\nGCS bucket: gs://{BUCKET_NAME}/")
print(f"  Images:   {GCS_IMAGE_PREFIX}/")
print(f"  Manifest: gs://{BUCKET_NAME}/automl/balanced_manifest.csv")
print(f"  CSV:      gs://{BUCKET_NAME}/balanced_dataset.csv")
print("=" * 60)